# 02 · Phase picking: volcanic-trained vs. general-purpose models

Deep-learning pickers are only as good as the data they were trained on. Most public
weights come from tectonic seismicity (California, Italy, global STEAD). Volcanic
earthquakes look different: shallow sources, strong path effects in altered rock, emergent
onsets, and **long-period (LP)** events with little high-frequency energy.

**VolPick** ([Zhong & Tan, 2024, *GRL*](https://doi.org/10.1029/2024GL108438)) retrained
PhaseNet and EQTransformer on **VCSEIS**, a compilation of waveforms from volcanic regions
(Alaska, Hawaii, Northern California) containing volcano-tectonic and long-period events,
regular earthquakes and noise. The models were then tested on data never used in training:
Cascade volcanoes and tectonic low-frequency earthquakes in the Nankai Trough. Aluto is
another such out-of-sample test. The weights are distributed through SeisBench under
the name `volpick`.

In this notebook we:

1. look inside the pickers: probability curves of several models on the same event;
2. pick the **whole day on all 11 stations** with six model/weight combinations;
3. compare them by number of picks, confidence, agreement and timing differences.

| Label | Architecture | Training data |
|---|---|---|
| `PN-original` | PhaseNet ([Zhu & Beroza, 2019](https://doi.org/10.1093/gji/ggy423)) | Northern California |
| `PN-instance` | PhaseNet | INSTANCE, Italy |
| `PN-stead` | PhaseNet | STEAD, global |
| `PN-volpick` | PhaseNet | VCSEIS: volcanic VT + LP, regular earthquakes, noise |
| `EQT-original` | EQTransformer ([Mousavi et al., 2020](https://doi.org/10.1038/s41467-020-17591-w)) | STEAD (original authors' weights) |
| `EQT-volpick` | EQTransformer | VCSEIS |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesfahiwet-yemane/SeismoML/blob/main/notebooks/02_phase_picking.ipynb)

In [ ]:
# Running on Google Colab? This cell installs everything and gets the data (2-4 minutes).
# On your own computer it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    repo = "/content/SeismoML"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tesfahiwet-yemane/SeismoML.git", repo], check=True)
    subprocess.run([sys.executable, f"{repo}/scripts/colab_setup.py"] + "--data --outputs".split(), check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seisbench
import seisbench.models as sbm
from obspy import UTCDateTime

import seismoml as sml
from seismoml import data, picking, plotting

plotting.use_style()
cfg = sml.load_config()
out_dir = cfg.outputs / "picks"
out_dir.mkdir(parents=True, exist_ok=True)

MODELS = [
    picking.ModelSpec("PhaseNet", "original"),
    picking.ModelSpec("PhaseNet", "instance"),
    picking.ModelSpec("PhaseNet", "stead"),
    picking.ModelSpec("PhaseNet", "volpick"),
    picking.ModelSpec("EQTransformer", "original"),
    picking.ModelSpec("EQTransformer", "volpick"),
]
THRESHOLD = 0.2      # common probability threshold for P and S
RECOMPUTE = False    # set True to ignore cached pick files in outputs/picks
print(f"SeisBench {seisbench.__version__}")

## 1 · What is available?

`list_pretrained()` queries the SeisBench model repository. Weights are downloaded once and
cached in `~/.seisbench`.

In [ ]:
for arch in ("PhaseNet", "EQTransformer"):
    names = getattr(sbm, arch).list_pretrained()
    print(f"{arch:14s}", ", ".join(names))
    assert "volpick" in names, f"volpick weights missing for {arch}"

In [ ]:
models = {}
for spec in MODELS:
    models[spec.label] = spec.load()
m = models["PN-volpick"]
print(f"PN-volpick: {m.in_samples} samples @ {m.sampling_rate} Hz, components {m.component_order}, "
      f"labels {m.labels}, norm = {getattr(m, 'norm', '?')}")
print(getattr(m, "weights_docstring", ""))

## 2 · Inside the black box: probability curves

The picker outputs a continuous probability for P, S (and noise or detection) at every
sample. Picks are the peaks above a threshold. Comparing the curves shows *how*
models disagree: timing offsets, missed phases, or confident false alarms.

In [ ]:
st_demo_day = data.load_waveforms(cfg.sds, cfg.network, cfg.demo_station, cfg.t0, cfg.t1)
trig = picking.sta_lta_triggers(st_demo_day.select(component="Z"))
onset = UTCDateTime(trig["onset"].iloc[0].timestamp())
win = st_demo_day.slice(onset - 90, onset + 90)
annotations = {label: model.annotate(win) for label, model in models.items()}
fig = plotting.multi_model_curves(win.slice(onset - 10, onset + 25), annotations,
                                  title=f"{cfg.network}.{cfg.demo_station} – strongest trigger of the day")
for ax in fig.axes[:3]:
    ax.set_xlim(0, 35)

## 3 · A whole day, the whole network

`classify` runs `annotate` and extracts picks. We use the same post-processing for every
model: windows overlap by 50 % and are averaged, and the outer 5 % of each window is discarded
("blinding"). See `seismoml.picking.default_inference_args`.

Picking 11 station-days with six models takes about 10–30 minutes on a laptop CPU. Results are
cached in `outputs/picks/`, so re-running the notebook is instant.

In [ ]:
def pick_network(model, stations):
    """Pick station by station (keeps memory low: one station-day at a time)."""
    tables = []
    for sta in stations:
        try:
            st = data.load_waveforms(cfg.sds, cfg.network, sta, cfg.t0, cfg.t1)
        except FileNotFoundError:
            print(f"  no data for {sta}, skipped")
            continue
        tables.append(picking.run_picker(model, st, threshold=THRESHOLD))
    tables = [t for t in tables if len(t)] or tables[:1]   # stations without picks add nothing
    return pd.concat(tables, ignore_index=True).sort_values("time").reset_index(drop=True)


picks = {}
runtime = {}
for label, model in models.items():
    path = out_dir / f"{label}.csv"
    if path.exists() and not RECOMPUTE:
        picks[label] = picking.read_picks(path)
        runtime[label] = np.nan
        print(f"{label:13s} {len(picks[label]):6d} picks (cached)")
        continue
    tic = time.perf_counter()
    df = pick_network(model, cfg.stations)
    runtime[label] = time.perf_counter() - tic
    picking.save_picks(df, path)
    picks[label] = df
    print(f"{label:13s} {len(df):6d} picks in {runtime[label]:6.0f} s")

In [ ]:
summary = pd.DataFrame({
    label: {
        "P picks": int((df.phase == "P").sum()),
        "S picks": int((df.phase == "S").sum()),
        "S/P ratio": (df.phase == "S").sum() / max((df.phase == "P").sum(), 1),
        "median P prob.": df.loc[df.phase == "P", "probability"].median(),
        "median S prob.": df.loc[df.phase == "S", "probability"].median(),
        "runtime (s)": runtime[label],
    } for label, df in picks.items()}).T
summary.style.format(precision=2)

In [ ]:
fig = plotting.probability_histograms(picks)

In [ ]:
fig = plotting.picks_per_hour(picks, phase="P")

**What to look for.** Diurnal cycles in the pick rate suggest cultural noise is being picked.
A model whose S/P ratio is far from the others is probably over- or under-calling S. Many
low-probability picks point to a model that is uncertain on these data. That is not always bad,
because the threshold can be adapted.

## 4 · Threshold dependence

A fixed threshold is not a fair comparison: models are calibrated differently. The number of
picks as a function of threshold shows each model's calibration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
colours = plotting.model_colours(picks)
for label, df in picks.items():
    curve = picking.threshold_curve(df, np.arange(THRESHOLD, 0.96, 0.05))
    for ax, ph in zip(axes, "PS"):
        ax.plot(curve["threshold"], curve[ph], color=colours[label], label=label)
for ax, ph in zip(axes, "PS"):
    ax.set(title=f"{ph} picks", xlabel="Threshold", yscale="log")
axes[0].set_ylabel("Picks surviving")
axes[1].legend(fontsize=8);

## 5 · Do the models agree?

For each pair of models, what fraction of model A's picks (row) does model B (column) also make
within ±0.3 s on the same station and phase?

In [ ]:
fig = plotting.matrix_heatmap(picking.agreement_matrix(picks, "P", 0.3),
                              title="P picks: row found by column (±0.3 s)")
fig = plotting.matrix_heatmap(picking.agreement_matrix(picks, "S", 0.3),
                              title="S picks: row found by column (±0.3 s)")

### Timing differences relative to PN-volpick

Where two models agree on a pick, how different are the times? A systematic offset means one
model consistently picks earlier, which matters for locations and for Vp/Vs estimates.

In [ ]:
ref_label = "PN-volpick"
matches = {label: picking.match_picks(picks[ref_label], df, tolerance=0.5)
           for label, df in picks.items() if label != ref_label}
fig = plotting.residual_histograms(matches, tolerance=0.5)

In [ ]:
pd.DataFrame({
    label: {
        f"{ph} matched": int((m.phase == ph).sum()) for ph in "PS"
    } | {
        f"{ph} median Δt (s)": m.loc[m.phase == ph, "residual"].median() for ph in "PS"
    } for label, m in matches.items()}).T.style.format(precision=3)

### Picks only one family finds

Which picks do the volcanic weights find that *no* general-purpose PhaseNet finds, and vice
versa? Look at the hourly distribution and a few examples to judge if they are real.

In [ ]:
generic = pd.concat([picks[k] for k in ("PN-original", "PN-instance", "PN-stead")])
generic = generic.sort_values("time").reset_index(drop=True)
m_vg = picking.match_picks(picks[ref_label], generic, tolerance=0.3)
only_volpick = picks[ref_label].drop(index=m_vg["ref_index"].unique())
print(f"{len(only_volpick)} of {len(picks[ref_label])} PN-volpick picks have no generic-PhaseNet counterpart")
only_volpick.groupby("phase")["probability"].describe()

In [ ]:
example = only_volpick[only_volpick.phase == "P"].sort_values("probability", ascending=False)
if len(example):
    ex = example.iloc[0]
    t_ex = UTCDateTime(ex["time"].timestamp())
    st_ex = data.load_waveforms(cfg.sds, cfg.network, ex["station"], t_ex - 60, t_ex + 60)
    ann_ex = {k: models[k].annotate(st_ex) for k in ("PN-volpick", "PN-instance", "PN-original")}
    fig = plotting.multi_model_curves(st_ex, ann_ex,
                                      title=f"P pick only found by PN-volpick: {ex['station']} {ex['time']:%H:%M:%S}")
    for ax in fig.axes[:3]:
        ax.set_xlim(50, 80)

## Take-aways

* Models disagree substantially on the same data. "Which picker?" is a scientific choice and
  needs to be validated, ideally against analyst picks (notebook 04).
* Pick counts alone are misleading. Calibration, timing bias and false-alarm behaviour all
  matter.
* The picks saved in `outputs/picks/` are the input to association (notebook 03).

## Exercises

1. Add `ModelSpec("PhaseNet", "ethz")` or another weight from the list in section 1. Where
   does it fall in the agreement matrix?
2. Repeat the comparison with `THRESHOLD = 0.5`. Which conclusions change?
3. **Ensemble picking:** keep only picks found by at least 3 of the 6 models (hint:
   `picking.match_picks`). How many remain?

**Next:** 03 · Association and location (`03_association.ipynb`)